In [1]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import math, os, json, regex as re, requests, random, gc, time
import torch
import torch.nn as nn
from torch.nn import functional as F

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(3407)
print("device:", device)

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision("high")


device: cuda


/usr/local/lib/python3.12/dist-packages/torch/backends/__init__.py:46: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:80.)
  self.setter(val)


In [2]:
def get_file(local_path, url):
    if not os.path.isfile(local_path):
        r = requests.get(url)
        r.raise_for_status()
        open(local_path, "wb").write(r.content)

def bytes_to_unicode():
    bs = list(range(ord("!"), ord("~")+1)) + list(range(ord("¡"), ord("¬")+1)) + list(range(ord("®"), ord("ÿ")+1))
    cs = bs[:]
    n = 0
    for b in range(256):
        if b not in bs:
            bs.append(b)
            cs.append(256 + n)
            n += 1
    cs = [chr(n) for n in cs]
    return dict(zip(bs, cs))

class GPT2BPE:
    def __init__(self):
        cache_dir = os.path.join(os.path.expanduser("~"), ".cache", "mini_bpe")
        os.makedirs(cache_dir, exist_ok=True)
        enc_f = os.path.join(cache_dir, "encoder.json")
        bpe_f = os.path.join(cache_dir, "vocab.bpe")
        get_file(enc_f, "https://openaipublic.blob.core.windows.net/gpt-2/models/124M/encoder.json")
        get_file(bpe_f, "https://openaipublic.blob.core.windows.net/gpt-2/models/124M/vocab.bpe")
        self.encoder = json.load(open(enc_f, "r"))
        #self.encoder is a dict like { "hello": 15496, "Ġthe": 464, ... }
        self.decoder = {v:k for k,v in self.encoder.items()}
        #load the BPE rules and make them fast to look up.
        merges = open(bpe_f, "r", encoding="utf-8").read().split("\n")[1:-1]
        merges = [tuple(m.split()) for m in merges]
        self.bpe_ranks = {m:i for i,m in enumerate(merges)}
        self.byte_encoder = bytes_to_unicode()
        self.byte_decoder = {v:k for k,v in self.byte_encoder.items()}
        self.pat = re.compile(r"""'s|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+""")
        self.cache = {}

    def _get_pairs(self, word):
        return set(zip(word, word[1:]))

    def _bpe(self, token):
        if token in self.cache: return self.cache[token]
        word = tuple(token)
        pairs = self._get_pairs(word)
        if not pairs: return token
        while True:
            bigram = min(pairs, key=lambda p: self.bpe_ranks.get(p, 1e10))
            if bigram not in self.bpe_ranks: break
            first, second = bigram
            new_word = []
            i = 0
            while i < len(word):
                try:
                    j = word.index(first, i)
                    new_word.extend(word[i:j]); i = j
                except ValueError:
                    new_word.extend(word[i:]); break
                if i < len(word)-1 and word[i] == first and word[i+1] == second:
                    new_word.append(first+second); i += 2
                else:
                    new_word.append(word[i]); i += 1
            word = tuple(new_word)
            if len(word) == 1: break
            pairs = self._get_pairs(word)
        out = " ".join(word)
        self.cache[token] = out
        return out

    def encode(self, text):
        bpe_idx = []
        for token in re.findall(self.pat, text):
            token_bytes = token.encode("utf-8")
            token_trans = "".join(self.byte_encoder[b] for b in token_bytes)
            token_merged = self._bpe(token_trans).split(" ")
            bpe_idx.extend(self.encoder[t] for t in token_merged)
        return bpe_idx

    def decode(self, ids):
        tokens = [self.decoder[i] for i in ids]
        text = "".join(tokens)
        text_bytes = bytearray([self.byte_decoder[c] for c in text])
        return text_bytes.decode("utf-8", errors="replace")

tokenizer = GPT2BPE()
vocab_size = 50257
print("vocab_size:", vocab_size)


vocab_size: 50257


In [3]:
URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
raw = requests.get(URL).text
ids = tokenizer.encode(raw)
data = torch.tensor(ids, dtype=torch.long)

n = int(0.9 * len(data))
train_ids = data[:n]
val_ids   = data[n:]

class BpeDataset:
    def __init__(self, ids, block_size):
        self.ids = ids
        self.block_size = block_size
    def get(self, batch_size):
        idx = torch.randint(0, len(self.ids) - self.block_size - 1, (batch_size,))
        x = torch.stack([self.ids[i:i+self.block_size] for i in idx])
        y = torch.stack([self.ids[i+1:i+self.block_size+1] for i in idx])
        return x.to(device), y.to(device)

print("train tokens:", len(train_ids), "val tokens:", len(val_ids))


train tokens: 304222 val tokens: 33803


In [4]:
class CFG:
    block_size = 128    # smaller so it's friendlier to Colab
    n_layer    = 4
    n_head     = 4
    n_embd     = 256
    dropout    = 0.1
    use_sinusoidal_pe = True

cfg = CFG()

def sinusoidal_pe(T, C, device):
    pe = torch.zeros(T, C, device=device)
    pos = torch.arange(0, T, device=device).unsqueeze(1)
    div = torch.exp(torch.arange(0, C, 2, device=device) * (-math.log(10000.0)/C))
    pe[:, 0::2] = torch.sin(pos * div)
    pe[:, 1::2] = torch.cos(pos * div)
    return pe

class LayerNorm(nn.Module):
    def __init__(self, C, eps=1e-5):
        super().__init__()
        self.w = nn.Parameter(torch.ones(C))
        self.b = nn.Parameter(torch.zeros(C))
        self.eps = eps
    def forward(self, x):
        m = x.mean(-1, keepdim=True)
        v = x.var(-1, keepdim=True, unbiased=False)
        return (x - m) / torch.sqrt(v + self.eps) * self.w + self.b

class CausalSelfAttention(nn.Module):
    def __init__(self, C, n_head, dropout, block_size):
        super().__init__()
        assert C % n_head == 0
        self.n_head = n_head
        self.head_dim = C // n_head
        self.qkv = nn.Linear(C, 3*C, bias=False)
        self.proj = nn.Linear(C, C, bias=False)
        self.attn_drop = nn.Dropout(dropout)
        self.resid_drop = nn.Dropout(dropout)
        self.register_buffer("mask", torch.tril(torch.ones(block_size, block_size)).view(1,1,block_size,block_size))
    def forward(self, x):
        B,T,C = x.shape
        qkv = self.qkv(x).view(B,T,3,self.n_head,self.head_dim)
        q,k,v = qkv.unbind(dim=2)
        q = q.transpose(1,2); k = k.transpose(1,2); v = v.transpose(1,2)
        att = (q @ k.transpose(-2,-1)) / math.sqrt(self.head_dim)
        att = att.masked_fill(self.mask[:,:,:T,:T] == 0, float("-inf"))
        att = att.softmax(-1)
        att = self.attn_drop(att)
        y = att @ v
        y = y.transpose(1,2).contiguous().view(B,T,C)
        return self.resid_drop(self.proj(y))

class MLP(nn.Module):
    def __init__(self, C, dropout):
        super().__init__()
        self.fc = nn.Linear(C, 4*C)
        self.proj = nn.Linear(4*C, C)
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        return self.drop(self.proj(F.gelu(self.fc(x))))

class Block(nn.Module):
    def __init__(self, C, n_head, dropout, block_size):
        super().__init__()
        self.ln1 = LayerNorm(C)
        self.attn = CausalSelfAttention(C, n_head, dropout, block_size)
        self.ln2 = LayerNorm(C)
        self.mlp  = MLP(C, dropout)
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

class GPT(nn.Module):
    def __init__(self, vocab_size, cfg: CFG):
        super().__init__()
        C = cfg.n_embd; T = cfg.block_size
        self.tok_emb = nn.Embedding(vocab_size, C)
        self.use_sinus = cfg.use_sinusoidal_pe
        if not self.use_sinus:
            self.pos_emb = nn.Embedding(T, C)
        self.drop = nn.Dropout(cfg.dropout)
        self.blocks = nn.ModuleList([Block(C, cfg.n_head, cfg.dropout, T) for _ in range(cfg.n_layer)])
        self.ln_f = LayerNorm(C)
        self.lm_head = nn.Linear(C, vocab_size, bias=False)
        self.lm_head.weight = self.tok_emb.weight

        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            if isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, mean=0.0, std=0.02)

        if self.use_sinus:
            self.register_buffer("pe", sinusoidal_pe(T, C, device="cpu"), persistent=False)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.tok_emb(idx)
        if self.use_sinus:
            pe = self.pe[:T, :].to(x.device)
            x = self.drop(x + pe)
        else:
            pos = torch.arange(T, device=x.device)
            x = self.drop(x + self.pos_emb(pos))
        for blk in self.blocks:
            x = blk(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens=128, temperature=0.8, top_k=40):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -cfg.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / max(temperature, 1e-6)
            if top_k:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float("inf")
            probs = F.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, next_id], dim=1)
        return idx

model = GPT(vocab_size, cfg).to(device)
print("model params:", sum(p.numel() for p in model.parameters())/1e6, "M")


model params: 16.021248 M


In [5]:
BATCH        = 4
ACCUM_STEPS  = 14     # effective ~56
MAX_STEPS    = 8000
LR           = 3e-4
WARMUP       = 200

train_ds = BpeDataset(train_ids, cfg.block_size)
val_ds   = BpeDataset(val_ids,   cfg.block_size)

opt = torch.optim.AdamW(model.parameters(), lr=LR, betas=(0.9,0.95), weight_decay=0.1)
scaler = torch.amp.GradScaler('cuda', enabled=(device=="cuda"))

def cosine_lr(step):
    if step < WARMUP: return step / max(1, WARMUP)
    p = (step - WARMUP) / max(1, (MAX_STEPS - WARMUP))
    return 0.5 * (1 + math.cos(math.pi * min(1.0, p)))

@torch.no_grad()
def eval_loss(ds, iters=5):
    model.eval(); losses = []
    for _ in range(iters):
        xb, yb = ds.get(BATCH)
        with torch.amp.autocast('cuda', enabled=(device=="cuda")):
            _, loss = model(xb, yb)
        losses.append(loss.item())
    model.train()
    return sum(losses)/len(losses)

best = float("inf")

for step in range(1, MAX_STEPS+1):
    lr_now = LR * cosine_lr(step)
    for g in opt.param_groups: g["lr"] = lr_now

    opt.zero_grad(set_to_none=True)
    loss_accum = 0.0
    for _ in range(ACCUM_STEPS):
        xb, yb = train_ds.get(BATCH)
        with torch.amp.autocast('cuda', enabled=(device=="cuda")):
            _, loss = model(xb, yb)
            loss = loss / ACCUM_STEPS
        scaler.scale(loss).backward()
        loss_accum += loss.item()

    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    scaler.step(opt); scaler.update()

    if step % 50 == 0:
        print(f"step {step:5d} | lr {lr_now:.2e} | loss {loss_accum:.4f}")

    if step % 200 == 0:
        tr = eval_loss(train_ds)
        va = eval_loss(val_ds)
        ppl = math.exp(va)
        print(f"[eval] step {step} | train {tr:.4f} | val {va:.4f} | ppl {ppl:.2f}")
        # save best
        if va < best:
            best = va
            torch.save(
                {"model": {k: v.cpu() for k,v in model.state_dict().items()}, "cfg": vars(cfg)},
                "/content/gpt_best.pt"
            )
            print("  -> saved /content/gpt_best.pt")


step    50 | lr 7.50e-05 | loss 10.5609
step   100 | lr 1.50e-04 | loss 9.8705
step   150 | lr 2.25e-04 | loss 8.6647
step   200 | lr 3.00e-04 | loss 7.2363
[eval] step 200 | train 7.2410 | val 7.2073 | ppl 1349.26
  -> saved /content/gpt_best.pt
step   250 | lr 3.00e-04 | loss 6.5939
step   300 | lr 3.00e-04 | loss 6.4615
step   350 | lr 3.00e-04 | loss 6.3791
step   400 | lr 3.00e-04 | loss 6.3892
[eval] step 400 | train 6.2934 | val 6.5911 | ppl 728.60
  -> saved /content/gpt_best.pt
step   450 | lr 2.99e-04 | loss 6.4180
step   500 | lr 2.99e-04 | loss 6.4681
step   550 | lr 2.99e-04 | loss 6.3188
step   600 | lr 2.98e-04 | loss 6.4197
[eval] step 600 | train 6.4462 | val 6.4308 | ppl 620.65
  -> saved /content/gpt_best.pt
step   650 | lr 2.98e-04 | loss 6.3510
step   700 | lr 2.97e-04 | loss 6.3287
step   750 | lr 2.96e-04 | loss 6.3539
step   800 | lr 2.96e-04 | loss 6.4175
[eval] step 800 | train 6.2742 | val 6.4290 | ppl 619.58
  -> saved /content/gpt_best.pt
step   850 | lr 2.

In [15]:
import os, gc, torch

ck = "/content/gpt_best.pt"
assert os.path.exists(ck), "No checkpoint found at /content/gpt_best.pt"

# only delete if they exist
if "opt" in globals():
    del opt
if "scaler" in globals():
    del scaler

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

sd = torch.load(ck, map_location="cpu")
model_cpu = GPT(vocab_size, cfg).to("cpu")
model_cpu.load_state_dict(sd["model"])
model_cpu.eval()

def sample(prompt, max_new_tokens=80, temperature=0.8, top_k=40):
    ids = torch.tensor([tokenizer.encode(prompt)], dtype=torch.long)
    with torch.inference_mode():
        out = model_cpu.generate(
            ids,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_k=top_k,
        )
    return tokenizer.decode(out[0].tolist())

# try it
print(sample("Juliette", max_new_tokens=80, temperature=0.8, top_k=40))


Juliette my for and thy him
The you in no, let,


KINGOLUS:
And will!
MENAN IUS:
O:
I have, the his;
HeARD:
LULI you me:
KINGCICHUS:
And, my, I he,

Why:
No:
IENES:
DC
